In [1]:
import xarray as xr

In [2]:
ds = xr.open_dataset("_surfdata_1x1_JP-Yoy_detailed_simyr2000_c230710.nc")
ds

<xarray.Dataset> Size: 10kB
Dimensions:                  (lsmlat: 1, lsmlon: 1, nlevsoi: 10, natpft: 15,
                              cft: 2, time: 12, lsmpft: 17, numurbl: 3,
                              ncl9: 3, ncl10: 1, ncl11: 3, ncl12: 1, ncl13: 3,
                              ncl14: 1, ncl15: 3, ncl16: 1, numrad: 2,
                              nlevurb: 10, nglcecp1: 11, nglcec: 10)
Coordinates:
  * natpft                   (natpft) int32 60B 0 1 2 3 4 5 ... 9 10 11 12 13 14
  * cft                      (cft) int32 8B 15 16
  * time                     (time) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
Dimensions without coordinates: lsmlat, lsmlon, nlevsoi, lsmpft, numurbl, ncl9,
                                ncl10, ncl11, ncl12, ncl13, ncl14, ncl15,
                                ncl16, numrad, nlevurb, nglcecp1, nglcec
Data variables: (12/78)
    mxsoil_color             int32 4B ...
    SOIL_COLOR               (lsmlat, lsmlon) int32 4B ...
    PCT_SAND                 (nlevsoi, lsmlat, lsmlon) float64 80B ...
    PCT_CLAY                 (nlevsoi, lsmlat, lsmlon) float64 80B ...
    ORGANIC                  (nlevsoi, lsmlat, lsmlon) float64 80B ...
    FMAX                     (lsmlat, lsmlon) float64 8B ...
    ...                       ...
    CONST_HARVEST_SH2        (lsmlat, lsmlon) float64 8B ...
    CONST_HARVEST_SH3        (lsmlat, lsmlon) float64 8B ...
    CONST_GRAZING            (lsmlat, lsmlon) float64 8B ...
    CONST_FERTNITRO_CFT      (cft, lsmlat, lsmlon) float64 16B ...
    UNREPRESENTED_PFT_LULCC  (natpft, lsmlat, lsmlon) float64 120B ...
    UNREPRESENTED_CFT_LULCC  (cft, lsmlat, lsmlon) float64 16B ...
Attributes: (12/49)
    Conventions:                          NCAR-CSM
    History_Log:                          created on: 05-25-21 16:56:00
    Logname:                              oleson
    Host:                                 r8i3n28
    Source:                               Community Land Model: CLM5
    Version:                              release-clm5.0.34-2-ga2989b04/glade...
    ...                                   ...
    map_topography_stats_file:            map_1km-merge-10min_HYDRO1K-merge-n...
    Soil_texture_raw_data_file_name:      mksrf_soitex.10level.c010119.nc
    Soil_color_raw_data_file_name:        mksrf_soilcolor_CMIP6_simyr2005.c17...
    Fmax_raw_data_file_name:              mksrf_fmax_3x3min_USGS_c120911.nc
    Organic_matter_raw_data_file_name:    mksrf_organic_10level_5x5min_ISRIC-...
    pft_override:                         TRUE

In [3]:
import numpy as np
ds['LT_BUILDING_MAX'] = ds['T_BUILDING_MIN']

# ref: https://www.env.go.jp/earth/ondanka/kateico2tokei/energy/detail/06/
ds['LT_BUILDING_MAX'].values = np.zeros(ds['LT_BUILDING_MAX'].values.shape) + 28 + 273.15
ds['T_BUILDING_MIN'].values = np.zeros(ds['T_BUILDING_MIN'].shape)  + 20 + 273.15
#ds['T_BUILDING_MIN'].values  = np.zeros(ds['T_BUILDING_MIN'].shape)  + 15 + 273.15

In [4]:
ds['T_BUILDING_MIN'].values, ds['LT_BUILDING_MAX'].values

(array([[[293.15]],
 
        [[293.15]],
 
        [[293.15]]]),
 array([[[301.15]],
 
        [[301.15]],
 
        [[301.15]]]))

In [5]:
ds.to_netcdf("surfdata_1x1_JP-Yoy_detailed_simyr2000_c230710.nc")

In [6]:
# 筛选含有维度numurbl的变量
variables_with_numurbl = [var for var in ds.data_vars if 'numurbl' in ds[var].dims]

for var in ds.data_vars:
    if 'EM_' in var:
        variables_with_numurbl.append(var)

variables_with_numurbl = variables_with_numurbl + ['PCT_SAND', 'PCT_CLAY']
print(variables_with_numurbl)

['CANYON_HWR', 'HT_ROOF', 'THICK_ROOF', 'THICK_WALL', 'T_BUILDING_MIN', 'WIND_HGT_CANYON', 'WTLUNIT_ROOF', 'WTROAD_PERV', 'WALL_TO_PLAN_AREA_RATIO', 'ALB_IMPROAD_DIR', 'ALB_IMPROAD_DIF', 'ALB_PERROAD_DIR', 'ALB_PERROAD_DIF', 'ALB_ROOF_DIR', 'ALB_ROOF_DIF', 'ALB_WALL_DIR', 'ALB_WALL_DIF', 'TK_ROOF', 'TK_WALL', 'TK_IMPROAD', 'CV_ROOF', 'CV_WALL', 'CV_IMPROAD', 'NLEV_IMPROAD', 'PCT_URBAN', 'LT_BUILDING_MAX', 'EM_IMPROAD', 'EM_PERROAD', 'EM_ROOF', 'EM_WALL', 'PCT_SAND', 'PCT_CLAY']


In [7]:
urb_dict = {}
for var in variables_with_numurbl:
    try:
        urb_dict[var] = ds[var].isel(numurbl=2).values.flatten().round(2).tolist()
        #urb_dict[var] = ds[var].values.flatten().tolist()[2]
    except Exception as e:
        if 'EM_' in var:
            urb_dict[var] = ds[var].values.flatten().round(2).tolist()[2]
        if 'PCT_SAND' in var:
            urb_dict[var] = ds[var].values.flatten().round(2).tolist()
        if 'PCT_CLAY' in var:
            urb_dict[var] = ds[var].values.flatten().round(2).tolist()
for key, value in urb_dict.items():
    print(f"{key}: {value}")  # 仅打印前10个值以简化输出

CANYON_HWR: [1.2]
HT_ROOF: [9.0]
THICK_ROOF: [0.26]
THICK_WALL: [0.32]
T_BUILDING_MIN: [293.15]
WIND_HGT_CANYON: [4.5]
WTLUNIT_ROOF: [0.41]
WTROAD_PERV: [0.14]
WALL_TO_PLAN_AREA_RATIO: [1.14]
ALB_IMPROAD_DIR: [0.13, 0.13]
ALB_IMPROAD_DIF: [0.13, 0.13]
ALB_PERROAD_DIR: [0.13, 0.13]
ALB_PERROAD_DIF: [0.13, 0.13]
ALB_ROOF_DIR: [0.23, 0.23]
ALB_ROOF_DIF: [0.23, 0.23]
ALB_WALL_DIR: [0.23, 0.23]
ALB_WALL_DIF: [0.23, 0.23]
TK_ROOF: [1.15, 0.19, 0.04, 0.04, 0.04, 0.7, 0.7, 0.7, 0.7, 0.7]
TK_WALL: [1.99, 4.29, 4.27, 3.29, 3.05, 0.76, 3.05, 3.29, 3.27, 1.08]
TK_IMPROAD: [1.67, 0.56, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
CV_ROOF: [1957200.0, 912000.0, 96600.0, 96600.0, 96600.0, 840000.0, 840000.0, 840000.0, 840000.0, 840000.0]
CV_WALL: [2118010.0, 2118500.0, 2133802.0, 95882.6, 777056.12, 776566.19, 777056.12, 134966.41, 121896.0, 617575.5]
CV_IMPROAD: [2060500.0, 1712300.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
NLEV_IMPROAD: [2]
PCT_URBAN: [100.0]
LT_BUILDING_MAX: [301.15]
EM_IMPROAD: 0.97
E